In [2]:

from pathlib import Path
import pandas as pd

BASE_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data"
)

TRUTHS_CLEAN_PATH = (
    BASE_PATH / "Project_Data" / "truths_clean.csv"
)

ADDITIONAL_PATH = (
    BASE_PATH / "Additional_Data" / "additional_truths.arff"
)

# Verify input files
for path in [TRUTHS_CLEAN_PATH, ADDITIONAL_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

# Load the cleaned main corpus
# Preserve external IDs as strings to avoid precision loss
df_truths = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    dtype={
        "external_id": "string",
        "url": "string"
    },
    low_memory=False
)

print("MAIN CORPUS")
print("-" * 40)
print("Shape:", df_truths.shape)

print("\nADDITIONAL ARFF")
print("-" * 40)
print("File:", ADDITIONAL_PATH)
print("Size (MB):", round(
    ADDITIONAL_PATH.stat().st_size / 1024**2, 2
))

print("\nMAIN CORPUS COLUMNS")
print(df_truths.columns.tolist())

print("\nMain corpus successfully loaded.")


MAIN CORPUS
----------------------------------------
Shape: (1038821, 13)

ADDITIONAL ARFF
----------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Additional_Data\additional_truths.arff
Size (MB): 13.13

MAIN CORPUS COLUMNS
['id', 'timestamp', 'time_scraped', 'is_retruth', 'is_reply', 'author', 'like_count', 'retruth_count', 'reply_count', 'text', 'external_id', 'url', 'truth_retruthed']

Main corpus successfully loaded.


In [3]:

# Load additional_truths.arff using the validated parsing strategy

additional_columns = [
    "id",
    "timestamp",
    "text",
    "url",
    "reply_count",
    "retruth_count",
    "like_count",
    "external_id",
    "author",
    "time_scraped"
]

records = []
data_started = False

with open(ADDITIONAL_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if line.lower() == "@data":
            data_started = True
            continue

        if not data_started or not line or line.startswith("%"):
            continue

        # First two attributes from the left
        left = line.split(",", 2)

        # Remaining seven attributes from the right
        right = left[2].rsplit(",", 7)

        row = [left[0], left[1]] + right

        if len(row) != len(additional_columns):
            raise ValueError(
                f"Invalid ARFF record with {len(row)} fields"
            )

        records.append(row)

df_additional = pd.DataFrame(
    records,
    columns=additional_columns
)

# Remove enclosing quotes without changing internal text
def remove_enclosing_quotes(value):
    value = value.strip()

    if len(value) >= 2 and value[0] == value[-1]:
        if value[0] in ("'", '"'):
            return value[1:-1]

    return value

for col in df_additional.columns:
    df_additional[col] = (
        df_additional[col].map(remove_enclosing_quotes)
    )

# Normalize identifiers without converting large IDs to floats
for col in ["id", "external_id"]:
    df_additional[col] = (
        df_additional[col].astype("string").str.strip()
    )

print("ADDITIONAL TRUTHS LOADING")
print("-" * 45)

print("Shape:", df_additional.shape)

print("\nIDENTIFIER VALIDATION")

id_match_mask = (
    df_additional["id"]
    .eq(df_additional["external_id"])
    .fillna(False)
)

print(
    "ID / external_id matches:",
    f"{id_match_mask.sum():,}"
)

print(
    "ID / external_id mismatches:",
    f"{(~id_match_mask).sum():,}"
)

print(
    "Duplicated external IDs:",
    f"{df_additional['external_id'].duplicated().sum():,}"
)

print("\nSAMPLE RECORDS")
display(df_additional.head())


ADDITIONAL TRUTHS LOADING
---------------------------------------------
Shape: (35222, 10)

IDENTIFIER VALIDATION
ID / external_id matches: 35,222
ID / external_id mismatches: 0
Duplicated external IDs: 0

SAMPLE RECORDS


,id,timestamp,text,url,reply_count,retruth_count,like_count,external_id,author,time_scraped
0,117323511508375196,2026-09-24 01:51:54,,https://truthsocial.com/@realDonaldTrump/11732...,792.0,1849.0,7259.0,117323511508375196,27.0,2026-09-20 14:00:00.000000
1,117323155721195611,2026-09-24 00:21:25,"The story that the White House, or me, want to...",https://truthsocial.com/@realDonaldTrump/11732...,1329.0,3534.0,12560.0,117323155721195611,27.0,2026-09-20 14:00:00.000000
2,117323148147220269,2026-09-24 00:19:29,BIG DAY FOR BOEING AND AMERICAN MANUFACTURING!...,https://truthsocial.com/@realDonaldTrump/11732...,510.0,2714.0,10677.0,117323148147220269,27.0,2026-09-20 14:00:00.000000
3,117322092055919832,2026-09-23 19:50:55,,https://truthsocial.com/@realDonaldTrump/11732...,1344.0,2740.0,11287.0,117322092055919832,27.0,2026-09-20 14:00:00.000000
4,117322065119445871,2026-09-23 19:44:04,"The Great Gary Player, Bryson DeChambeau, Mika...",https://truthsocial.com/@realDonaldTrump/11732...,505.0,2156.0,10284.0,117322065119445871,27.0,2026-09-20 14:00:00.000000


In [4]:

# Compare external identifiers across the two datasets

main_external_ids = (
    df_truths["external_id"]
    .astype("string")
    .str.strip()
)

additional_external_ids = (
    df_additional["external_id"]
    .astype("string")
    .str.strip()
)

# Remove missing identifiers from the comparison
main_id_set = set(main_external_ids.dropna())
additional_id_set = set(additional_external_ids.dropna())

# Identify overlapping and exclusive posts
common_ids = main_id_set & additional_id_set

only_main_ids = main_id_set - additional_id_set

only_additional_ids = additional_id_set - main_id_set

# Summarize the results
overlap_summary = pd.DataFrame({
    "Category": [
        "Common posts",
        "Only in main corpus",
        "Only in additional dataset"
    ],
    "Number of posts": [
        len(common_ids),
        len(only_main_ids),
        len(only_additional_ids)
    ]
})

print("CROSS-DATASET EXTERNAL ID COMPARISON")
print("-" * 45)

display(overlap_summary)

print(
    "\nPercentage of additional IDs already in main corpus:",
    f"{len(common_ids) / len(additional_id_set) * 100:.2f}%"
)

# Identify additional records eligible for integration
new_additional_mask = (
    additional_external_ids.isin(only_additional_ids)
)

df_additional_new = df_additional.loc[
    new_additional_mask
].copy()

print(
    "\nAdditional records not found in main corpus:",
    f"{len(df_additional_new):,}"
)

print("\nSAMPLE NEW RECORDS")
display(
    df_additional_new[
        ["id", "external_id", "author", "timestamp"]
    ].head(10)
)


CROSS-DATASET EXTERNAL ID COMPARISON
---------------------------------------------


,Category,Number of posts
0,Common posts,32
1,Only in main corpus,892844
2,Only in additional dataset,35190



Percentage of additional IDs already in main corpus: 0.09%

Additional records not found in main corpus: 35,190

SAMPLE NEW RECORDS


,id,external_id,author,timestamp
0,117323511508375196,117323511508375196,27.0,2026-09-24 01:51:54
1,117323155721195611,117323155721195611,27.0,2026-09-24 00:21:25
2,117323148147220269,117323148147220269,27.0,2026-09-24 00:19:29
3,117322092055919832,117322092055919832,27.0,2026-09-23 19:50:55
4,117322065119445871,117322065119445871,27.0,2026-09-23 19:44:04
5,117320581360547091,117320581360547091,27.0,2026-09-23 13:26:43
6,117320509296515188,117320509296515188,27.0,2026-09-23 13:08:24
7,117320385350427395,117320385350427395,27.0,2026-09-23 12:36:52
8,117318371569831036,117318371569831036,27.0,2026-09-23 04:04:44
9,117318370203435194,117318370203435194,27.0,2026-09-23 04:04:24


In [5]:

# Investigate the 32 overlapping external identifiers

main_overlap = df_truths.loc[
    main_external_ids.isin(common_ids),
    [
        "id", "external_id", "author", "timestamp",
        "text", "like_count", "retruth_count", "reply_count"
    ]
].copy()

additional_overlap = df_additional.loc[
    additional_external_ids.isin(common_ids),
    [
        "id", "external_id", "author", "timestamp",
        "text", "like_count", "retruth_count", "reply_count"
    ]
].copy()

# Normalize external IDs
for df in [main_overlap, additional_overlap]:
    df["external_id"] = (
        df["external_id"].astype("string").str.strip()
    )

# Rename columns before the merge
main_overlap = main_overlap.rename(
    columns={
        col: f"{col}_main"
        for col in main_overlap.columns
        if col != "external_id"
    }
)

additional_overlap = additional_overlap.rename(
    columns={
        col: f"{col}_additional"
        for col in additional_overlap.columns
        if col != "external_id"
    }
)

# Match records by platform identifier
overlap_comparison = main_overlap.merge(
    additional_overlap,
    on="external_id",
    how="inner",
    validate="many_to_one"
)

# Compare authors
overlap_comparison["same_author"] = (
    pd.to_numeric(
        overlap_comparison["author_main"], errors="coerce"
    ).eq(
        pd.to_numeric(
            overlap_comparison["author_additional"], errors="coerce"
        )
    )
)

# Compare publication timestamps
overlap_comparison["same_timestamp"] = (
    pd.to_datetime(
        overlap_comparison["timestamp_main"], errors="coerce"
    ).eq(
        pd.to_datetime(
            overlap_comparison["timestamp_additional"], errors="coerce"
        )
    )
)

# Compare text
overlap_comparison["same_text"] = (
    overlap_comparison["text_main"].astype("string")
    .str.strip()
    .eq(
        overlap_comparison["text_additional"]
        .astype("string").str.strip()
    )
    .fillna(False)
)

print("OVERLAPPING POSTS VALIDATION")
print("-" * 45)

print("Shared external IDs:", len(common_ids))
print("Matched record pairs:", len(overlap_comparison))

print("\nCONSISTENCY SUMMARY")
for col in ["same_author", "same_timestamp", "same_text"]:
    print(f"\n{col}")
    print(overlap_comparison[col].value_counts(dropna=False))

print("\nSAMPLE OVERLAPPING RECORDS")
display(
    overlap_comparison[
        [
            "external_id",
            "id_main",
            "id_additional",
            "same_author",
            "same_timestamp",
            "same_text"
        ]
    ].head(15)
)

print("\nSAMPLE INCONSISTENT RECORDS")
display(
    overlap_comparison.loc[
        ~(
            overlap_comparison["same_author"]
            & overlap_comparison["same_timestamp"]
            & overlap_comparison["same_text"]
        ),
        [
            "external_id",
            "author_main",
            "author_additional",
            "timestamp_main",
            "timestamp_additional",
            "same_text"
        ]
    ].head(10)
)


OVERLAPPING POSTS VALIDATION
---------------------------------------------
Shared external IDs: 32
Matched record pairs: 35

CONSISTENCY SUMMARY

same_author
same_author
True     33
False     2
Name: count, dtype: int64

same_timestamp
same_timestamp
False    35
Name: count, dtype: int64

same_text
same_text
False    21
True     14
Name: count, dtype: Int64

SAMPLE OVERLAPPING RECORDS


,external_id,id_main,id_additional,same_author,same_timestamp,same_text
0,109110471933646660,895498,109110471933646660,True,False,False
1,109110378607366886,895500,109110378607366886,True,False,True
2,109105126982139161,895502,109105126982139161,True,False,True
3,109105107965281984,895506,109105107965281984,True,False,True
4,109102258492682585,895508,109102258492682585,True,False,False
5,109111035929718455,895496,109111035929718455,True,False,False
6,109096508463033991,895514,109096508463033991,True,False,False
7,109093444861506227,895517,109093444861506227,True,False,True
8,109102004910960727,895510,109102004910960727,True,False,False
9,109082720069591446,895525,109082720069591446,True,False,False



SAMPLE INCONSISTENT RECORDS


,external_id,author_main,author_additional,timestamp_main,timestamp_additional,same_text
0,109110471933646660,27,27.0,2022-10-04,2022-10-04 14:27:55,False
1,109110378607366886,27,27.0,2022-10-04,2022-10-04 14:04:11,True
2,109105126982139161,27,27.0,2022-10-03,2022-10-03 15:48:38,True
3,109105107965281984,27,27.0,2022-10-03,2022-10-03 15:43:48,True
4,109102258492682585,27,27.0,2022-10-02,2022-10-03 03:39:08,False
5,109111035929718455,27,27.0,2022-10-03,2022-10-04 16:51:21,False
6,109096508463033991,27,27.0,2022-10-01,2022-10-02 03:16:50,False
7,109093444861506227,27,27.0,2022-10-01,2022-10-01 14:17:43,True
8,109102004910960727,27,27.0,2022-10-02,2022-10-03 02:34:39,False
9,109082720069591446,27,27.0,2022-09-29,2022-09-29 16:50:15,False


In [6]:

# Prepare the additional records for schema integration

df_additional_aligned = df_additional_new.copy()

# Normalize textual attributes
text_columns = [
    "timestamp",
    "time_scraped",
    "text",
    "url",
    "external_id"
]

for col in text_columns:
    df_additional_aligned[col] = (
        df_additional_aligned[col]
        .astype("string")
        .str.strip()
    )

# Convert missing markers to proper missing values
missing_markers = ["", "?", "nan", "NaN", "None"]

for col in ["text", "url", "external_id"]:
    df_additional_aligned[col] = (
        df_additional_aligned[col]
        .replace(missing_markers, pd.NA)
    )

# Convert date columns
for col in ["timestamp", "time_scraped"]:
    df_additional_aligned[col] = pd.to_datetime(
        df_additional_aligned[col],
        errors="coerce"
    )

# Convert numerical attributes
numeric_columns = [
    "author",
    "like_count",
    "retruth_count",
    "reply_count"
]

for col in numeric_columns:
    values = pd.to_numeric(
        df_additional_aligned[col]
        .astype("string")
        .str.strip()
        .replace(["", "?", "nan", "NaN"], pd.NA),
        errors="coerce"
    )

    # Check for non-integer numerical values
    fractional_mask = values.notna() & values.mod(1).ne(0)

    if fractional_mask.any():
        raise ValueError(
            f"Non-integer values found in {col}"
        )

    df_additional_aligned[col] = values.astype("Int64")

# Preserve platform identifiers
df_additional_aligned["external_id"] = (
    df_additional_aligned["external_id"].astype("string")
)

# Add attributes absent from the ARFF dataset
df_additional_aligned["is_retruth"] = pd.Series(
    pd.NA,
    index=df_additional_aligned.index,
    dtype="boolean"
)

df_additional_aligned["is_reply"] = pd.Series(
    pd.NA,
    index=df_additional_aligned.index,
    dtype="boolean"
)

df_additional_aligned["truth_retruthed"] = pd.Series(
    pd.NA,
    index=df_additional_aligned.index,
    dtype="Int64"
)

print("ADDITIONAL DATASET SCHEMA HARMONIZATION")
print("-" * 45)

print("Records:", f"{len(df_additional_aligned):,}")

print("\nDATA TYPES")
display(df_additional_aligned.dtypes.to_frame("Data type"))

print("\nMISSING VALUES")
display(
    df_additional_aligned.isna()
    .sum()
    .to_frame("Missing values")
)

# Check temporal inconsistencies before integration
future_scrape_mask = (
    df_additional_aligned["timestamp"].notna()
    & df_additional_aligned["time_scraped"].notna()
    & (
        df_additional_aligned["timestamp"]
        > df_additional_aligned["time_scraped"]
    )
)

print(
    "\nPublication after scraping:",
    f"{future_scrape_mask.sum():,}"
)

print("\nSchema harmonization completed.")


ADDITIONAL DATASET SCHEMA HARMONIZATION
---------------------------------------------
Records: 35,190

DATA TYPES


,Data type
id,string
timestamp,datetime64[us]
text,string
url,string
reply_count,Int64
retruth_count,Int64
like_count,Int64
external_id,string
author,Int64
time_scraped,datetime64[us]



MISSING VALUES


,Missing values
id,0
timestamp,0
text,7259
url,0
reply_count,9
retruth_count,9
like_count,9
external_id,0
author,0
time_scraped,0



Publication after scraping: 42

Schema harmonization completed.


In [7]:

# Investigate temporal inconsistencies in the additional dataset

temporal_anomalies = df_additional_aligned.loc[
    future_scrape_mask,
    ["id", "timestamp", "time_scraped", "author"]
].copy()

# Calculate the temporal difference
temporal_anomalies["difference_hours"] = (
    temporal_anomalies["timestamp"]
    - temporal_anomalies["time_scraped"]
).dt.total_seconds() / 3600

# Check whether timestamps belong to the same calendar day
temporal_anomalies["same_calendar_day"] = (
    temporal_anomalies["timestamp"].dt.normalize()
    == temporal_anomalies["time_scraped"].dt.normalize()
)

print("ADDITIONAL TRUTHS TEMPORAL VALIDATION")
print("-" * 45)

print("Total inconsistencies:", len(temporal_anomalies))

print("\nTIME DIFFERENCE (HOURS)")
display(
    temporal_anomalies["difference_hours"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nCALENDAR DAY COMPARISON")
print(
    "Same calendar day:",
    temporal_anomalies["same_calendar_day"].sum()
)

print(
    "Different calendar days:",
    (~temporal_anomalies["same_calendar_day"]).sum()
)

print("\nSCRAPING TIMESTAMPS INVOLVED")
display(
    temporal_anomalies["time_scraped"]
    .value_counts()
    .head(10)
)

print("\nSAMPLE TEMPORAL ANOMALIES")
display(
    temporal_anomalies.sort_values(
        "difference_hours",
        ascending=False
    ).head(15)
)


ADDITIONAL TRUTHS TEMPORAL VALIDATION
---------------------------------------------
Total inconsistencies: 42

TIME DIFFERENCE (HOURS)


count    42.000000
mean     37.835364
std      25.506219
min       2.370000
25%      15.110347
50%      28.721944
75%      62.064236
90%      77.105528
95%      82.100917
99%      83.246697
max      83.865000
Name: difference_hours, dtype: float64


CALENDAR DAY COMPARISON
Same calendar day: 5
Different calendar days: 37

SCRAPING TIMESTAMPS INVOLVED


time_scraped
2026-09-20 14:00:00    42
Name: count, dtype: int64


SAMPLE TEMPORAL ANOMALIES


,id,timestamp,time_scraped,author,difference_hours,same_calendar_day
0,117323511508375196,2026-09-24 01:51:54,2026-09-20 14:00:00,27,83.865000,False
1,117323155721195611,2026-09-24 00:21:25,2026-09-20 14:00:00,27,82.356944,False
2,117323148147220269,2026-09-24 00:19:29,2026-09-20 14:00:00,27,82.324722,False
3,117322092055919832,2026-09-23 19:50:55,2026-09-20 14:00:00,27,77.848611,False
4,117322065119445871,2026-09-23 19:44:04,2026-09-20 14:00:00,27,77.734444,False
5,117320581360547091,2026-09-23 13:26:43,2026-09-20 14:00:00,27,71.445278,False
6,117320509296515188,2026-09-23 13:08:24,2026-09-20 14:00:00,27,71.140000,False
7,117320385350427395,2026-09-23 12:36:52,2026-09-20 14:00:00,27,70.614444,False
8,117318371569831036,2026-09-23 04:04:44,2026-09-20 14:00:00,27,62.078889,False
9,117318370203435194,2026-09-23 04:04:24,2026-09-20 14:00:00,27,62.073333,False


In [8]:

# Handle inconsistent scraping timestamps in additional Truths

invalid_scraping_mask = (
    df_additional_aligned["timestamp"].notna()
    & df_additional_aligned["time_scraped"].notna()
    & (
        df_additional_aligned["timestamp"]
        > df_additional_aligned["time_scraped"]
    )
)

print("SCRAPING TIMESTAMP CLEANING")
print("-" * 45)

print(
    "Inconsistent timestamps before cleaning:",
    f"{invalid_scraping_mask.sum():,}"
)

# Preserve the records but invalidate unreliable scraping times
df_additional_aligned.loc[
    invalid_scraping_mask,
    "time_scraped"
] = pd.NaT

# Validate the transformation
remaining_inconsistencies = (
    df_additional_aligned["timestamp"].notna()
    & df_additional_aligned["time_scraped"].notna()
    & (
        df_additional_aligned["timestamp"]
        > df_additional_aligned["time_scraped"]
    )
)

assert remaining_inconsistencies.sum() == 0
assert len(df_additional_aligned) == 35190

print(
    "Inconsistent timestamps after cleaning:",
    f"{remaining_inconsistencies.sum():,}"
)

print(
    "Missing scraping timestamps:",
    f"{df_additional_aligned['time_scraped'].isna().sum():,}"
)

print("\nScraping timestamp cleaning completed.")


SCRAPING TIMESTAMP CLEANING
---------------------------------------------
Inconsistent timestamps before cleaning: 42
Inconsistent timestamps after cleaning: 0
Missing scraping timestamps: 42

Scraping timestamp cleaning completed.


In [9]:

import numpy as np

# Create a separate copy for integration
df_additional_integrated = df_additional_aligned.copy()

# Ensure main corpus IDs are valid integers
main_internal_ids = pd.to_numeric(
    df_truths["id"],
    errors="raise"
).astype("int64")

assert main_internal_ids.is_unique
assert main_internal_ids.notna().all()

# Assign new internal primary keys
max_existing_id = main_internal_ids.max()

new_internal_ids = np.arange(
    max_existing_id + 1,
    max_existing_id + len(df_additional_integrated) + 1,
    dtype="int64"
)

df_additional_integrated["id"] = new_internal_ids

# Align columns with the original corpus
original_columns = df_truths.columns.tolist()

df_additional_integrated = df_additional_integrated[
    original_columns
].copy()

# Ensure the same data types for key attributes
df_additional_integrated["external_id"] = (
    df_additional_integrated["external_id"].astype("string")
)

df_truths["external_id"] = (
    df_truths["external_id"].astype("string")
)

# Integrate the two datasets
df_integrated = pd.concat(
    [df_truths, df_additional_integrated],
    ignore_index=True
)

print("TRUTHS CORPUS INTEGRATION")
print("-" * 45)

print("Main corpus records:", f"{len(df_truths):,}")
print("Additional records:", f"{len(df_additional_integrated):,}")
print("Integrated corpus records:", f"{len(df_integrated):,}")

print("\nSCHEMA")
print("Total columns:", df_integrated.shape[1])
print("Columns match original:", (
    df_integrated.columns.tolist() == original_columns
))

print("\nPRIMARY KEY VALIDATION")
print("Missing IDs:", df_integrated["id"].isna().sum())
print("Duplicated IDs:", df_integrated["id"].duplicated().sum())

# Integrity checks
assert len(df_integrated) == (
    len(df_truths) + len(df_additional_integrated)
)

assert df_integrated["id"].is_unique
assert df_integrated["id"].notna().all()
assert df_integrated.columns.tolist() == original_columns

print("\nIntegration validation passed.")

print("\nSAMPLE INTEGRATED RECORDS")
display(df_integrated.tail(10))


TRUTHS CORPUS INTEGRATION
---------------------------------------------
Main corpus records: 1,038,821
Additional records: 35,190
Integrated corpus records: 1,074,011

SCHEMA
Total columns: 13
Columns match original: True

PRIMARY KEY VALIDATION
Missing IDs: 0
Duplicated IDs: 0

Integration validation passed.

SAMPLE INTEGRATED RECORDS


,id,timestamp,time_scraped,is_retruth,is_reply,author,like_count,retruth_count,reply_count,text,external_id,url,truth_retruthed
1074001,2418705,2022-05-11 00:51:29,2026-09-20 14:00:00,<NA>,<NA>,27,81805,29577,3985,"“TRUMPISM,” as some call AMERICA FIRST, is ver...",108280564237440718,https://truthsocial.com/@realDonaldTrump/10828...,<NA>
1074002,2418706,2022-05-09 01:10:08,2026-09-20 14:00:00,<NA>,<NA>,27,78536,34997,7276,<NA>,108269312986461850,https://truthsocial.com/@realDonaldTrump/10826...,<NA>
1074003,2418707,2022-05-08 20:01:20,2026-09-20 14:00:00,<NA>,<NA>,27,34343,7729,1536,<NA>,108268098736318851,https://truthsocial.com/@realDonaldTrump/10826...,<NA>
1074004,2418708,2022-05-07 18:30:10,2026-09-20 14:00:00,<NA>,<NA>,27,46703,13537,1409,<NA>,108262077899921897,https://truthsocial.com/@realDonaldTrump/10826...,<NA>
1074005,2418709,2022-05-07 15:51:06,2026-09-20 14:00:00,<NA>,<NA>,27,0,0,0,RT @DevinNunesIt’s FAKENEWS Friday! Anyone wh...,108261452443854009,https://truthsocial.com/@realDonaldTrump/10826...,<NA>
1074006,2418710,2022-05-02 00:57:26,2026-09-20 14:00:00,<NA>,<NA>,27,0,0,0,RT @RSBNPresident Trump calls on Joe Biden to ...,108229626843484243,https://truthsocial.com/@realDonaldTrump/10822...,<NA>
1074007,2418711,2022-05-02 00:53:17,2026-09-20 14:00:00,<NA>,<NA>,27,0,0,0,RT @RSBNMASSIVE turnout for @realDonaldTrump i...,108229610518342758,https://truthsocial.com/@realDonaldTrump/10822...,<NA>
1074008,2418712,2022-04-30 16:41:10,2026-09-20 14:00:00,<NA>,<NA>,27,0,0,0,RT @DevinNunesWhat a beautiful day to wake up ...,108222013152768428,https://truthsocial.com/@realDonaldTrump/10822...,<NA>
1074009,2418713,2022-04-30 12:37:05,2026-09-20 14:00:00,<NA>,<NA>,27,0,0,0,RT @catturd2Joe Biden is going to fly gas guzz...,108221053343991942,https://truthsocial.com/@realDonaldTrump/10822...,<NA>
1074010,2418714,2022-04-28 21:29:28,2026-09-20 14:00:00,<NA>,<NA>,27,410837,123196,60886,I’M BACK! #COVFEFE,108211822140637685,https://truthsocial.com/@realDonaldTrump/10821...,<NA>


In [10]:

# Fix mixed datetime parsing before final integration

df_truths_ready = df_truths.copy()
df_additional_ready = df_additional_integrated.copy()

# Parse mixed datetime formats
for col in ["timestamp", "time_scraped"]:
    df_truths_ready[col] = pd.to_datetime(
        df_truths_ready[col],
        format="mixed",
        errors="coerce"
    )

    df_additional_ready[col] = pd.to_datetime(
        df_additional_ready[col],
        format="mixed",
        errors="coerce"
    )

# Validate timestamp preservation BEFORE concatenation
print("TEMPORAL DATA VALIDATION")
print("-" * 45)

for col in ["timestamp", "time_scraped"]:
    print(f"\n{col}")
    print(
        "Main corpus missing:",
        f"{df_truths_ready[col].isna().sum():,}"
    )
    print(
        "Additional records missing:",
        f"{df_additional_ready[col].isna().sum():,}"
    )

assert df_truths_ready["timestamp"].notna().all()
assert df_truths_ready["time_scraped"].notna().all()

assert df_additional_ready["timestamp"].notna().all()
assert df_additional_ready["time_scraped"].isna().sum() == 42

# Harmonize remaining data types
for col in ["text", "url", "external_id"]:
    df_truths_ready[col] = df_truths_ready[col].astype("string")
    df_additional_ready[col] = df_additional_ready[col].astype("string")

integer_columns = [
    "id", "author", "like_count",
    "retruth_count", "reply_count", "truth_retruthed"
]

for col in integer_columns:
    df_truths_ready[col] = pd.to_numeric(
        df_truths_ready[col], errors="raise"
    ).astype("Int64")

    df_additional_ready[col] = pd.to_numeric(
        df_additional_ready[col], errors="raise"
    ).astype("Int64")

for col in ["is_retruth", "is_reply"]:
    df_truths_ready[col] = df_truths_ready[col].astype("boolean")
    df_additional_ready[col] = df_additional_ready[col].astype("boolean")

# Rebuild the integrated corpus
df_integrated = pd.concat(
    [df_truths_ready, df_additional_ready],
    ignore_index=True
)

# Verify the expected result
assert df_integrated["timestamp"].isna().sum() == 0
assert df_integrated["time_scraped"].isna().sum() == 42
assert df_integrated["id"].is_unique
assert df_integrated.shape == (1074011, 13)

print("\nINTEGRATED DATASET")
print("Shape:", df_integrated.shape)

print("\nFINAL TEMPORAL MISSING VALUES")
display(
    df_integrated[["timestamp", "time_scraped"]]
    .isna()
    .sum()
    .to_frame("Missing values")
)

print("\nTemporal correction validated successfully.")


TEMPORAL DATA VALIDATION
---------------------------------------------

timestamp
Main corpus missing: 0
Additional records missing: 0

time_scraped
Main corpus missing: 0
Additional records missing: 42

INTEGRATED DATASET
Shape: (1074011, 13)

FINAL TEMPORAL MISSING VALUES


,Missing values
timestamp,0
time_scraped,42



Temporal correction validated successfully.


In [11]:

# Final validation of the integrated corpus

main_count = len(df_truths_ready)
additional_count = len(df_additional_ready)

print("FINAL INTEGRATION VALIDATION")
print("-" * 45)

# Validate dimensions and schema
assert len(df_integrated) == main_count + additional_count
assert df_integrated.shape[1] == 13

# Validate primary key
assert df_integrated["id"].notna().all()
assert df_integrated["id"].is_unique

# Verify preservation of the aligned main corpus
pd.testing.assert_frame_equal(
    df_integrated.iloc[:main_count].reset_index(drop=True),
    df_truths_ready.reset_index(drop=True),
    check_dtype=True
)

# Verify internal IDs do not overlap
main_ids = set(df_truths_ready["id"])
additional_ids = set(df_additional_ready["id"])

assert main_ids.isdisjoint(additional_ids)

# Verify external identifiers do not overlap
main_external = set(
    df_truths_ready["external_id"].dropna()
)

additional_external = set(
    df_additional_ready["external_id"].dropna()
)

assert main_external.isdisjoint(additional_external)

print("Original records preserved: True")
print("Primary keys unique: True")
print("External ID overlap: 0")

print("\nFINAL DATASET")
print("Records:", f"{len(df_integrated):,}")
print("Columns:", df_integrated.shape[1])

print("\nMISSING VALUES")
display(
    df_integrated.isna()
    .sum()
    .to_frame("Missing values")
)

print("\nFinal integration validation passed.")


FINAL INTEGRATION VALIDATION
---------------------------------------------
Original records preserved: True
Primary keys unique: True
External ID overlap: 0

FINAL DATASET
Records: 1,074,011
Columns: 13

MISSING VALUES


,Missing values
id,0
timestamp,0
time_scraped,42
is_retruth,35190
is_reply,35190
author,0
like_count,9
retruth_count,9
reply_count,9
text,7437



Final integration validation passed.


In [12]:
# ============================================================
# STEP 11: BUILD ARFF-TO-INTEGRATED TRUTH ID MAPPING
# ============================================================

# We need to preserve the correspondence between the
# original ARFF identifiers and the internal primary keys
# assigned in the integrated Truths corpus.

# ------------------------------------------------------------
# 1. Map newly integrated ARFF posts
# ------------------------------------------------------------

# Both DataFrames preserve the same original row order
assert df_additional_new.index.equals(df_additional_ready.index)

new_mapping = pd.DataFrame({
    "arff_truth_id": (
        df_additional_new["id"]
        .astype("string")
        .to_numpy()
    ),
    "integrated_truth_id": (
        df_additional_ready["id"]
        .astype("Int64")
        .to_numpy()
    )
})

new_mapping["mapping_status"] = "new_post"

# ------------------------------------------------------------
# 2. Investigate ARFF posts already present in main corpus
# ------------------------------------------------------------

shared_arff = (
    df_additional.loc[
        df_additional["external_id"].isin(common_ids),
        ["id", "external_id"]
    ]
    .rename(columns={"id": "arff_truth_id"})
    .copy()
)

main_candidates = df_truths_ready.loc[
    df_truths_ready["external_id"].isin(common_ids),
    ["external_id", "id", "author", "timestamp", "is_retruth"]
].copy()

# Count distinct internal IDs per external identifier
candidate_counts = (
    main_candidates
    .groupby("external_id")["id"]
    .nunique()
)

# Only identifiers with exactly one internal ID
# can be mapped without ambiguity
unique_external_ids = candidate_counts[
    candidate_counts == 1
].index

unique_main_mapping = (
    main_candidates.loc[
        main_candidates["external_id"].isin(unique_external_ids),
        ["external_id", "id"]
    ]
    .drop_duplicates(subset=["external_id"])
    .rename(columns={"id": "integrated_truth_id"})
)

shared_mapping = shared_arff.merge(
    unique_main_mapping,
    on="external_id",
    how="left",
    validate="one_to_one"
)

shared_mapping["mapping_status"] = "shared_unique"

shared_mapping.loc[
    shared_mapping["integrated_truth_id"].isna(),
    "mapping_status"
] = "shared_ambiguous"

shared_mapping = shared_mapping[
    ["arff_truth_id", "integrated_truth_id", "mapping_status"]
]

# ------------------------------------------------------------
# 3. Combine mappings
# ------------------------------------------------------------

truth_id_mapping = pd.concat(
    [new_mapping, shared_mapping],
    ignore_index=True
)

truth_id_mapping["arff_truth_id"] = (
    truth_id_mapping["arff_truth_id"].astype("string")
)

truth_id_mapping["integrated_truth_id"] = (
    pd.to_numeric(
        truth_id_mapping["integrated_truth_id"],
        errors="coerce"
    ).astype("Int64")
)

# ------------------------------------------------------------
# 4. Validate mapping
# ------------------------------------------------------------

assert len(truth_id_mapping) == len(df_additional)

assert truth_id_mapping["arff_truth_id"].is_unique

assert set(truth_id_mapping["arff_truth_id"]) == set(
    df_additional["id"]
)

mapped_ids = truth_id_mapping[
    "integrated_truth_id"
].dropna()

assert mapped_ids.isin(df_integrated["id"]).all()

print("ARFF-TO-INTEGRATED ID MAPPING")
print("-" * 50)

print("Total ARFF records:", len(df_additional))
print("Mapping records:", len(truth_id_mapping))

print("\nMAPPING STATUS")
display(
    truth_id_mapping["mapping_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nUNRESOLVED AMBIGUITIES")

ambiguous_ids = truth_id_mapping.loc[
    truth_id_mapping["mapping_status"] == "shared_ambiguous",
    "arff_truth_id"
]

display(
    main_candidates.loc[
        main_candidates["external_id"].isin(ambiguous_ids)
    ].sort_values(["external_id", "id"])
)

print("\nSAMPLE MAPPING")
display(truth_id_mapping.head(10))

# ------------------------------------------------------------
# 5. Save mapping for use in the Events notebook
# ------------------------------------------------------------

MAPPING_PATH = (
    BASE_PATH
    / "Project_Data"
    / "arff_truth_id_mapping.csv"
)

truth_id_mapping.to_csv(
    MAPPING_PATH,
    index=False
)

print("\nMapping saved to:", MAPPING_PATH)
print("Mapping validation completed.")

ARFF-TO-INTEGRATED ID MAPPING
--------------------------------------------------
Total ARFF records: 35222
Mapping records: 35222

MAPPING STATUS


,status,records
0,new_post,35190
1,shared_unique,29
2,shared_ambiguous,3



UNRESOLVED AMBIGUITIES


,external_id,id,author,timestamp,is_retruth
134672,109017355982286163,895593,27,2022-09-17 00:00:00,True
134686,109017355982286163,895595,145,2022-09-17 00:00:00,False
134684,109027164115408705,895584,27,2022-09-19 00:00:00,True
134683,109027164115408705,895587,559,2022-09-19 00:00:00,False
950542,113008796172112928,1943937,27,2024-08-23 01:43:00,False
951384,113008796172112928,1948317,27,2024-08-23 01:43:00,False



SAMPLE MAPPING


,arff_truth_id,integrated_truth_id,mapping_status
0,117323511508375196,2383525,new_post
1,117323155721195611,2383526,new_post
2,117323148147220269,2383527,new_post
3,117322092055919832,2383528,new_post
4,117322065119445871,2383529,new_post
5,117320581360547091,2383530,new_post
6,117320509296515188,2383531,new_post
7,117320385350427395,2383532,new_post
8,117318371569831036,2383533,new_post
9,117318370203435194,2383534,new_post



Mapping saved to: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\arff_truth_id_mapping.csv
Mapping validation completed.


In [13]:
# ============================================================
# STEP 12: INVESTIGATE AMBIGUOUS ARFF ID MAPPINGS
# ============================================================

# Select the three unresolved ARFF identifiers
ambiguous_ids = set(
    truth_id_mapping.loc[
        truth_id_mapping["mapping_status"] == "shared_ambiguous",
        "arff_truth_id"
    ].astype("string")
)

# Retrieve all candidate matches from the previous analysis
ambiguous_details = overlap_comparison.loc[
    overlap_comparison["external_id"]
    .astype("string")
    .isin(ambiguous_ids)
].copy()

# Recover the retruth flag from the main corpus
retruth_lookup = (
    df_truths_ready
    .set_index("id")["is_retruth"]
)

ambiguous_details["is_retruth_main"] = (
    ambiguous_details["id_main"].map(retruth_lookup)
)

# Compare calendar dates (main timestamps may have
# lower precision than ARFF timestamps)
ambiguous_details["same_calendar_date"] = (
    pd.to_datetime(
        ambiguous_details["timestamp_main"],
        errors="coerce"
    ).dt.date.eq(
        pd.to_datetime(
            ambiguous_details["timestamp_additional"],
            errors="coerce"
        ).dt.date
    ).fillna(False)
)

# Prepare readable previews of the post content
for source in ["main", "additional"]:
    ambiguous_details[f"text_{source}_preview"] = (
        ambiguous_details[f"text_{source}"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.slice(0, 200)
    )

# Display comparison
comparison_columns = [
    "external_id",
    "id_main",
    "id_additional",
    "author_main",
    "author_additional",
    "is_retruth_main",
    "same_author",
    "same_calendar_date",
    "same_text",
    "text_main_preview",
    "text_additional_preview"
]

print("AMBIGUOUS ARFF RECORD COMPARISON")
print("-" * 55)

display(
    ambiguous_details[comparison_columns]
    .sort_values(["external_id", "id_main"])
)

print("\nCANDIDATES PER ARFF IDENTIFIER")

display(
    ambiguous_details.groupby("external_id")
    .agg(
        candidates=("id_main", "nunique"),
        matching_author=("same_author", "sum"),
        matching_text=("same_text", "sum"),
        matching_date=("same_calendar_date", "sum")
    )
    .reset_index()
)

print("\nInvestigation completed. No mappings modified.")

AMBIGUOUS ARFF RECORD COMPARISON
-------------------------------------------------------


,external_id,id_main,id_additional,author_main,author_additional,is_retruth_main,same_author,same_calendar_date,same_text,text_main_preview,text_additional_preview
17,109017355982286163,895593,109017355982286163,27,27.0,True,True,False,False,<,RT @NewsMaxJUST IN: Former President Donald Tr...
26,109017355982286163,895595,109017355982286163,145,27.0,False,False,False,False,<,RT @NewsMaxJUST IN: Former President Donald Tr...
24,109027164115408705,895584,109027164115408705,27,27.0,True,True,True,False,<,RT @MelaniaTrumpRIP Her Majesty Queen Elizabet...
23,109027164115408705,895587,109027164115408705,559,27.0,False,False,True,False,<,RT @MelaniaTrumpRIP Her Majesty Queen Elizabet...
31,113008796172112928,1943937,113008796172112928,27,27.0,False,True,False,False,Thank you to #BrianKempGA for all of your help...,Thank you to #BrianKempGA for all of your help...
32,113008796172112928,1948317,113008796172112928,27,27.0,False,True,False,False,Thank you to #BrianKempGA for all of your help...,Thank you to #BrianKempGA for all of your help...



CANDIDATES PER ARFF IDENTIFIER


,external_id,candidates,matching_author,matching_text,matching_date
0,109017355982286163,2,1,0,0
1,109027164115408705,2,1,0,2
2,113008796172112928,2,2,0,0



Investigation completed. No mappings modified.


In [14]:
# ============================================================
# STEP 13: RESOLVE UNAMBIGUOUS AUTHOR/RETRUTH MATCHES
# ============================================================

# Select candidates consistent with:
# 1. The ARFF author
# 2. A retruth in the main corpus
# 3. The "RT @" structure of the ARFF text

candidate_matches = ambiguous_details.loc[
    ambiguous_details["same_author"].fillna(False)
    & ambiguous_details["is_retruth_main"].fillna(False)
    & ambiguous_details["text_additional"]
        .astype("string")
        .str.strip()
        .str.match(r"(?i)^RT\s+@", na=False)
].copy()

# Keep only identifiers with exactly one eligible candidate
candidate_counts = (
    candidate_matches
    .groupby("external_id")["id_main"]
    .nunique()
)

resolvable_ids = candidate_counts[
    candidate_counts == 1
].index

resolved_mapping = (
    candidate_matches.loc[
        candidate_matches["external_id"].isin(resolvable_ids),
        ["external_id", "id_main"]
    ]
    .drop_duplicates()
    .rename(columns={
        "external_id": "arff_truth_id",
        "id_main": "integrated_truth_id"
    })
    .copy()
)

resolved_mapping["arff_truth_id"] = (
    resolved_mapping["arff_truth_id"].astype("string")
)

resolved_mapping["integrated_truth_id"] = (
    resolved_mapping["integrated_truth_id"].astype("Int64")
)

# Update only the unresolved entries
for row in resolved_mapping.itertuples(index=False):

    mask = (
        (truth_id_mapping["arff_truth_id"] == row.arff_truth_id)
        & (truth_id_mapping["mapping_status"] == "shared_ambiguous")
    )

    assert mask.sum() == 1

    truth_id_mapping.loc[
        mask, "integrated_truth_id"
    ] = row.integrated_truth_id

    truth_id_mapping.loc[
        mask, "mapping_status"
    ] = "shared_inferred_author_rt"

# Validate results
assert truth_id_mapping["arff_truth_id"].is_unique
assert len(truth_id_mapping) == len(df_additional)

assert (
    truth_id_mapping["integrated_truth_id"]
    .dropna()
    .isin(df_integrated["id"])
    .all()
)

print("UPDATED ARFF ID MAPPING")
print("-" * 50)

display(
    truth_id_mapping["mapping_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nINFERRED MAPPINGS")
display(resolved_mapping)

print("\nREMAINING AMBIGUITIES")
display(
    truth_id_mapping.loc[
        truth_id_mapping["mapping_status"] == "shared_ambiguous"
    ]
)

# Save the updated mapping
truth_id_mapping.to_csv(
    MAPPING_PATH,
    index=False
)

print("\nUpdated mapping saved to:", MAPPING_PATH)

UPDATED ARFF ID MAPPING
--------------------------------------------------


,status,records
0,new_post,35190
1,shared_unique,29
2,shared_inferred_author_rt,2
3,shared_ambiguous,1



INFERRED MAPPINGS


,arff_truth_id,integrated_truth_id
17,109017355982286163,895593
24,109027164115408705,895584



REMAINING AMBIGUITIES


,arff_truth_id,integrated_truth_id,mapping_status
35194,113008796172112928,<NA>,shared_ambiguous



Updated mapping saved to: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\arff_truth_id_mapping.csv


In [15]:
# ============================================================
# STEP 14: INSPECT REMAINING AMBIGUOUS RECORDS
# ============================================================

remaining_ids = set(
    truth_id_mapping.loc[
        truth_id_mapping["mapping_status"] == "shared_ambiguous",
        "arff_truth_id"
    ]
)

remaining_details = ambiguous_details.loc[
    ambiguous_details["external_id"]
    .astype("string")
    .isin(remaining_ids)
].copy()

print("REMAINING AMBIGUOUS RECORDS")
print("-" * 60)

for _, row in remaining_details.iterrows():

    print("\n" + "=" * 60)

    print("ARFF external ID:", row["external_id"])
    print("Main internal ID:", row["id_main"])

    print("\nAUTHOR")
    print("Main:", row["author_main"])
    print("ARFF:", row["author_additional"])

    print("\nTIMESTAMP")
    print("Main:", row["timestamp_main"])
    print("ARFF:", row["timestamp_additional"])

    main_text = str(row["text_main"])
    arff_text = str(row["text_additional"])

    print("\nTEXT LENGTH")
    print("Main:", len(main_text))
    print("ARFF:", len(arff_text))

    print("\nMAIN TEXT")
    print(repr(main_text))

    print("\nARFF TEXT")
    print(repr(arff_text))

    print("\nEXACT TEXT MATCH:", row["same_text"])

REMAINING AMBIGUOUS RECORDS
------------------------------------------------------------

ARFF external ID: 113008796172112928
Main internal ID: 1943937

AUTHOR
Main: 27
ARFF: 27.0

TIMESTAMP
Main: 2024-08-23 01:43:00
ARFF: 2024-08-23 01:43:02

TEXT LENGTH
Main: 283
ARFF: 279

MAIN TEXT
'Thank you to #BrianKempGA for all of your help and support in Georgia, where a win is so important to the success of our Party and, most importantly, our Country.&nbsp;  I look forward to working with you, your team, and all of my friends in Georgia to help MAKE AMERICA GREAT AGAIN!'

ARFF TEXT
'Thank you to #BrianKempGA for all of your help and support in Georgia, where a win is so important to the success of our Party and, most importantly, our Country.\\xa0I look forward to working with you, your team, and all of my friends in Georgia to help MAKE AMERICA GREAT AGAIN!'

EXACT TEXT MATCH: False

ARFF external ID: 113008796172112928
Main internal ID: 1948317

AUTHOR
Main: 27
ARFF: 27.0

TIMESTAMP
Main

In [16]:
# ============================================================
# STEP 15: NORMALIZED TEXT COMPARISON FOR AMBIGUOUS POSTS
# ============================================================

import html
import re
import unicodedata

# Normalize HTML entities, Unicode characters and whitespace
def normalize_post_text(value):

    if pd.isna(value):
        return pd.NA

    text = html.unescape(str(value))

    # Handle both actual and literal escaped non-breaking spaces
    text = text.replace("\u00a0", " ")
    text = text.replace("\\xa0", " ")

    # Normalize Unicode representation
    text = unicodedata.normalize("NFKC", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Select only unresolved ARFF identifiers
remaining_ids = set(
    truth_id_mapping.loc[
        truth_id_mapping["mapping_status"] == "shared_ambiguous",
        "arff_truth_id"
    ].astype("string")
)

remaining = ambiguous_details.loc[
    ambiguous_details["external_id"]
    .astype("string")
    .isin(remaining_ids)
].copy()

# Normalize both text versions
remaining["text_main_normalized"] = (
    remaining["text_main"].apply(normalize_post_text)
)

remaining["text_additional_normalized"] = (
    remaining["text_additional"].apply(normalize_post_text)
)

# Compare normalized content
remaining["same_normalized_text"] = (
    remaining["text_main_normalized"]
    .eq(remaining["text_additional_normalized"])
    .fillna(False)
)

# Parse mixed timestamp formats consistently
remaining["timestamp_main_parsed"] = pd.to_datetime(
    remaining["timestamp_main"],
    format="mixed",
    errors="coerce",
    utc=True
)

remaining["timestamp_additional_parsed"] = pd.to_datetime(
    remaining["timestamp_additional"],
    format="mixed",
    errors="coerce",
    utc=True
)

# Calculate absolute timestamp difference
remaining["time_difference_seconds"] = (
    remaining["timestamp_main_parsed"]
    - remaining["timestamp_additional_parsed"]
).abs().dt.total_seconds()

# Recheck calendar dates
remaining["same_calendar_date_fixed"] = (
    remaining["timestamp_main_parsed"].dt.date
    == remaining["timestamp_additional_parsed"].dt.date
)

# Prepare concise comparison
comparison = remaining[
    [
        "external_id",
        "id_main",
        "author_main",
        "author_additional",
        "same_author",
        "is_retruth_main",
        "time_difference_seconds",
        "same_calendar_date_fixed",
        "same_text",
        "same_normalized_text"
    ]
].copy()

print("NORMALIZED AMBIGUOUS RECORD COMPARISON")
print("-" * 60)

display(comparison)

print("\nNORMALIZED TEXT MATCHES:",
      int(remaining["same_normalized_text"].sum()))

print("TOTAL CANDIDATES:", len(remaining))

print("\nNo records or mappings modified.")

NORMALIZED AMBIGUOUS RECORD COMPARISON
------------------------------------------------------------


,external_id,id_main,author_main,author_additional,same_author,is_retruth_main,time_difference_seconds,same_calendar_date_fixed,same_text,same_normalized_text
31,113008796172112928,1943937,27,27.0,True,False,2.0,True,False,True
32,113008796172112928,1948317,27,27.0,True,False,2.0,True,False,True



NORMALIZED TEXT MATCHES: 2
TOTAL CANDIDATES: 2

No records or mappings modified.


In [17]:
# ============================================================
# STEP 16: FINALIZE AMBIGUOUS ARFF ID MAPPING
# ============================================================

# Identify candidates satisfying all matching criteria
valid_candidates = remaining.loc[
    remaining["same_author"].fillna(False)
    & remaining["same_normalized_text"].fillna(False)
    & remaining["same_calendar_date_fixed"].fillna(False)
    & remaining["time_difference_seconds"].le(5)
    & remaining["is_retruth_main"].eq(False).fillna(False)
].copy()

# Verify that all remaining candidates meet the criteria
assert len(valid_candidates) == len(remaining)

# Verify that only one ambiguous ARFF identifier remains
assert len(remaining_ids) == 1

# Select a deterministic canonical internal ID
canonical_id = int(valid_candidates["id_main"].min())
arff_id = next(iter(remaining_ids))

# Preserve the alternative internal IDs for auditing
alternative_ids = sorted(
    set(valid_candidates["id_main"].astype(int))
    - {canonical_id}
)

# Update only the remaining ambiguous mapping
mask = (
    (truth_id_mapping["arff_truth_id"] == arff_id)
    & (truth_id_mapping["mapping_status"] == "shared_ambiguous")
)

assert mask.sum() == 1

truth_id_mapping.loc[
    mask, "integrated_truth_id"
] = canonical_id

truth_id_mapping.loc[
    mask, "mapping_status"
] = "shared_canonical_normalized"

# Create an audit record
canonical_audit = pd.DataFrame([{
    "arff_truth_id": arff_id,
    "canonical_truth_id": canonical_id,
    "alternative_truth_ids": ",".join(map(str, alternative_ids)),
    "resolution_rule": (
        "Same author, normalized text, calendar date, "
        "non-retruth status and timestamp difference <= 5s; "
        "minimum internal ID selected"
    )
}])

# Validate final mapping
assert len(truth_id_mapping) == len(df_additional)
assert truth_id_mapping["arff_truth_id"].is_unique

assert truth_id_mapping["integrated_truth_id"].notna().all()

assert truth_id_mapping["integrated_truth_id"].isin(
    df_integrated["id"]
).all()

assert (
    truth_id_mapping["mapping_status"] == "shared_ambiguous"
).sum() == 0

# Save updated mapping
truth_id_mapping.to_csv(
    MAPPING_PATH,
    index=False
)

# Save audit information
AUDIT_PATH = (
    BASE_PATH
    / "Project_Data"
    / "arff_canonical_id_audit.csv"
)

canonical_audit.to_csv(
    AUDIT_PATH,
    index=False
)

print("FINAL ARFF-TO-INTEGRATED ID MAPPING")
print("-" * 55)

display(
    truth_id_mapping["mapping_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nCANONICAL ID RESOLUTION")
display(canonical_audit)

print("\nFINAL VALIDATION")
print("Total mappings:", len(truth_id_mapping))
print("Missing integrated IDs:",
      truth_id_mapping["integrated_truth_id"].isna().sum())
print("Unresolved ambiguities:",
      (truth_id_mapping["mapping_status"] == "shared_ambiguous").sum())

print("\nMapping saved to:", MAPPING_PATH)
print("Audit saved to:", AUDIT_PATH)
print("\nFinal mapping validation passed.")

FINAL ARFF-TO-INTEGRATED ID MAPPING
-------------------------------------------------------


,status,records
0,new_post,35190
1,shared_unique,29
2,shared_inferred_author_rt,2
3,shared_canonical_normalized,1



CANONICAL ID RESOLUTION


,arff_truth_id,canonical_truth_id,alternative_truth_ids,resolution_rule
0,113008796172112928,1943937,1948317,"Same author, normalized text, calendar date, n..."



FINAL VALIDATION
Total mappings: 35222
Missing integrated IDs: 0
Unresolved ambiguities: 0

Mapping saved to: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\arff_truth_id_mapping.csv
Audit saved to: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\arff_canonical_id_audit.csv

Final mapping validation passed.


## Trying to fill structural MV is_retruth, is_reply, truth_retruthed...

In [18]:

# Investigate textual indicators of retruths and replies
# without modifying the integrated dataset

additional_text = (
    df_additional_ready["text"]
    .astype("string")
    .fillna("")
    .str.strip()
)

# Possible explicit repost indicators
rt_pattern = additional_text.str.contains(
    r"(?i)^RT\s+@\w+",
    regex=True,
    na=False
)

# Posts beginning with a user mention
mention_pattern = additional_text.str.contains(
    r"^@\w+",
    regex=True,
    na=False
)

# Missing or empty text
empty_text_mask = additional_text.eq("")

print("POST TYPE INFERENCE ANALYSIS")
print("-" * 45)

print("Total additional posts:", f"{len(additional_text):,}")

print(
    "Posts starting with 'RT @username':",
    f"{rt_pattern.sum():,}"
)

print(
    "Posts starting with '@username':",
    f"{mention_pattern.sum():,}"
)

print(
    "Posts without text:",
    f"{empty_text_mask.sum():,}"
)

print("\nSAMPLE POTENTIAL RETRUTHS")
display(
    df_additional_ready.loc[
        rt_pattern,
        ["id", "external_id", "text", "timestamp"]
    ].head(10)
)

print("\nSAMPLE POTENTIAL REPLIES")
display(
    df_additional_ready.loc[
        mention_pattern,
        ["id", "external_id", "text", "timestamp"]
    ].head(10)
)


POST TYPE INFERENCE ANALYSIS
---------------------------------------------
Total additional posts: 35,190
Posts starting with 'RT @username': 5,448
Posts starting with '@username': 24
Posts without text: 7,259

SAMPLE POTENTIAL RETRUTHS


,id,external_id,text,timestamp
16,2383541,117312560106447861,RT @Schoenlawhttps://www.wsj.com/opinion/ameri...,2026-09-22 03:26:49
57,2383582,117293485474940230,RT @realDonaldTrumpTRUMP Approval Rating in th...,2026-09-18 18:35:53
97,2383622,117284234108021902,RT @realDonaldTrump,2026-09-17 03:23:08
203,2383728,117238467019560891,RT @realDonaldTrumpThe Failing New York Magazi...,2026-09-09 01:23:58
206,2383731,117238345561593751,RT @realDonaldTrumpThe Failing New York Magazi...,2026-09-09 00:53:04
337,2383862,117213740097315077,RT @realDonaldTrumpI am pleased to report that...,2026-09-04 16:35:35
338,2383863,117213738409833358,"RT @MichaelCohen212""The Weaponized State""They ...",2026-09-04 16:35:09
348,2383873,117210071109792571,RT @realDonaldTrumpThe Missouri Supreme Court ...,2026-09-04 01:02:31
349,2383874,117210070942098771,RT @realDonaldTrumpI am pleased to announce th...,2026-09-04 01:02:28
389,2383914,117197478170575070,"RT @realDonaldTrumpThe United States is, as we...",2026-09-01 19:39:58



SAMPLE POTENTIAL REPLIES


,id,external_id,text,timestamp
8113,2391638,115022215816175004,"@DonaldJTrumpJr: ""We hit rock bottoms as a Cou...",2025-08-13 15:42:08
8119,2391644,115021897738089560,"@EricTrump: ""My father has a Cabinet that\'s u...",2025-08-13 14:21:15
9200,2392725,114712684243270785,@IStandWithTrump47 Depart of Treasury just ann...,2025-06-19 23:44:11
11192,2394717,114042789770444354,@RepMattGaetz: “From cutting bloated budgets t...,2025-02-21 16:21:13
13431,2396956,113290104652637540,@RSBN is doing a phenomenal job covering today...,2024-10-11 18:03:29
14447,2397972,113114683012097704,"@EricTrump: ""6 weeks ago, they were laughing a...",2024-09-10 18:31:27
15291,2398811,112984587451161172,@MarkLevinShow: “Dems nominated '2 of the most...,2024-08-18 19:06:27
16551,2400071,112724782504134192,@TheLeeGreenwood and I are championing Faith a...,2024-07-03 21:54:38
18861,2402380,112281357744662497,@LaraTrump: “It\'s not Donald Trump that\'s tr...,2024-04-16 14:25:53
19544,2403063,112150416019309205,@MonicaCrowley: “Donald Trump essentially had ...,2024-03-24 11:25:41


In [19]:
# ============================================================
# STEP 17: EXPORT THE INTEGRATED TRUTHS CORPUS
# ============================================================

TRUTHS_INTEGRATED_PATH = (
    BASE_PATH / "Project_Data" / "truths_integrated.csv"
)

# ------------------------------------------------------------
# 1. Final validation before export
# ------------------------------------------------------------

assert len(df_integrated) == 1074011

assert df_integrated["id"].notna().all()
assert df_integrated["id"].is_unique

assert len(df_integrated.columns) == 13

assert df_integrated["timestamp"].notna().all()

# Verify that all mapped ARFF IDs exist in the corpus
assert truth_id_mapping[
    "integrated_truth_id"
].isin(df_integrated["id"]).all()

print("PRE-EXPORT VALIDATION")
print("-" * 50)

print("Total records:", len(df_integrated))
print("Total columns:", len(df_integrated.columns))
print("Duplicated primary keys:",
      df_integrated["id"].duplicated().sum())

# ------------------------------------------------------------
# 2. Export integrated corpus
# ------------------------------------------------------------

df_integrated.to_csv(
    TRUTHS_INTEGRATED_PATH,
    index=False
)

# ------------------------------------------------------------
# 3. Reload exported dataset
# ------------------------------------------------------------

exported_truths = pd.read_csv(
    TRUTHS_INTEGRATED_PATH,
    dtype={
        "id": "Int64",
        "external_id": "string",
        "author": "Int64",
        "truth_retruthed": "Int64"
    },
    low_memory=False
)

# ------------------------------------------------------------
# 4. Validate exported dataset
# ------------------------------------------------------------

assert len(exported_truths) == len(df_integrated)

assert exported_truths["id"].notna().all()
assert exported_truths["id"].is_unique

assert exported_truths["id"].equals(
    df_integrated["id"].reset_index(drop=True)
)

assert list(exported_truths.columns) == list(
    df_integrated.columns
)

# Validate mapped identifiers after reloading
assert truth_id_mapping[
    "integrated_truth_id"
].isin(exported_truths["id"]).all()

# ------------------------------------------------------------
# 5. Export report
# ------------------------------------------------------------

print("\nINTEGRATED TRUTHS EXPORT")
print("-" * 50)

print("File:", TRUTHS_INTEGRATED_PATH)
print("Records:", len(exported_truths))
print("Columns:", len(exported_truths.columns))

print("\nFILE SIZE (MB)")
print(
    round(
        TRUTHS_INTEGRATED_PATH.stat().st_size / 1024**2,
        2
    )
)

print("\nPRIMARY KEY VALIDATION")
print("Missing IDs:", exported_truths["id"].isna().sum())
print("Duplicated IDs:", exported_truths["id"].duplicated().sum())

print("\nARFF MAPPING VALIDATION")
print(
    "Mapped IDs found in exported corpus:",
    truth_id_mapping["integrated_truth_id"]
    .isin(exported_truths["id"])
    .sum()
)

print("\nExport validation passed.")

PRE-EXPORT VALIDATION
--------------------------------------------------
Total records: 1074011
Total columns: 13
Duplicated primary keys: 0

INTEGRATED TRUTHS EXPORT
--------------------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\truths_integrated.csv
Records: 1074011
Columns: 13

FILE SIZE (MB)
383.65

PRIMARY KEY VALIDATION
Missing IDs: 0
Duplicated IDs: 0

ARFF MAPPING VALIDATION
Mapped IDs found in exported corpus: 35222

Export validation passed.


In [20]:
# ============================================================
# STEP 18: FINAL CROSS-DATASET REFERENTIAL INTEGRITY
# ============================================================

EVENTS_INTEGRATED_PATH = (
    BASE_PATH / "Project_Data" / "events_integrated.csv"
)

if not EVENTS_INTEGRATED_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {EVENTS_INTEGRATED_PATH}"
    )

# ------------------------------------------------------------
# 1. Load only the identifiers from the exported CSV files
# ------------------------------------------------------------

truths_keys = pd.read_csv(
    TRUTHS_INTEGRATED_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)

events_keys = pd.read_csv(
    EVENTS_INTEGRATED_PATH,
    usecols=["event_id", "truth_id_integrated"],
    dtype={
        "event_id": "Int64",
        "truth_id_integrated": "Int64"
    }
)

# ------------------------------------------------------------
# 2. Validate primary and foreign keys
# ------------------------------------------------------------

assert truths_keys["id"].notna().all()
assert truths_keys["id"].is_unique

assert events_keys["event_id"].notna().all()
assert events_keys["event_id"].is_unique

valid_truth_ids = set(truths_keys["id"].dropna())

matched_mask = (
    events_keys["truth_id_integrated"]
    .isin(valid_truth_ids)
)

unmatched_mask = (
    events_keys["truth_id_integrated"].notna()
    & ~matched_mask
)

missing_mask = (
    events_keys["truth_id_integrated"].isna()
)

# ------------------------------------------------------------
# 3. Generate referential integrity report
# ------------------------------------------------------------

integrity_report = pd.DataFrame({
    "check": [
        "Truths primary keys",
        "Events primary keys",
        "Matched foreign keys",
        "Unmatched foreign keys",
        "Missing foreign keys"
    ],
    "records": [
        len(truths_keys),
        len(events_keys),
        int(matched_mask.sum()),
        int(unmatched_mask.sum()),
        int(missing_mask.sum())
    ]
})

print("FINAL CROSS-DATASET REFERENTIAL INTEGRITY")
print("-" * 55)

display(integrity_report)

print("\nREFERENTIAL INTEGRITY RATE")

integrity_rate = (
    matched_mask.sum() / len(events_keys) * 100
)

print(f"{integrity_rate:.2f}%")

# ------------------------------------------------------------
# 4. Final assertions
# ------------------------------------------------------------

assert len(truths_keys) == 1074011
assert len(events_keys) == 716643

assert matched_mask.all()
assert unmatched_mask.sum() == 0
assert missing_mask.sum() == 0

print("\nAll cross-dataset integrity checks passed.")

FINAL CROSS-DATASET REFERENTIAL INTEGRITY
-------------------------------------------------------


,check,records
0,Truths primary keys,1074011
1,Events primary keys,716643
2,Matched foreign keys,716643
3,Unmatched foreign keys,0
4,Missing foreign keys,0



REFERENTIAL INTEGRITY RATE
100.00%

All cross-dataset integrity checks passed.
